# Advanced NLP Tasks

## Overview

This notebook covers advanced NLP tasks that go beyond basic text classification and NER. These form the backbone of **information extraction**, **document understanding**, and **natural language understanding** pipelines.

### Task Hierarchy

```
Text
 ├── Syntactic Analysis
 │   ├── POS Tagging            → What role does each word play?
 │   ├── Dependency Parsing     → How words relate syntactically
 │   └── Constituency Parsing   → Phrase-structure trees
 │
 ├── Semantic Analysis
 │   ├── Coreference Resolution → Which mentions refer to same entity?
 │   ├── Relation Extraction    → What relations exist between entities?
 │   └── Event Extraction       → What events happened? Who/where/when?
 │
 ├── Text Manipulation
 │   ├── Style Transfer         → Change style while preserving meaning
 │   ├── Text Simplification    → Make text easier to understand
 │   └── Paraphrase Mining      → Find semantically equivalent texts
 │
 └── Document Understanding
     ├── LayoutLM / Donut       → Documents with visual layout
     └── Scientific PDF parsing → Structure from PDFs
```

In [1]:
# Setup
# pip install spacy transformers sentence-transformers datasets
# python -m spacy download en_core_web_sm
# python -m spacy download en_core_web_trf  # transformer-based

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

print("Imports ready. Install spacy models with:")
print("  python -m spacy download en_core_web_sm")
print("  python -m spacy download en_core_web_trf")

Imports ready. Install spacy models with:
  python -m spacy download en_core_web_sm
  python -m spacy download en_core_web_trf


## 1. Part-of-Speech (POS) Tagging

POS tagging assigns grammatical categories (noun, verb, adjective, etc.) to each token.

### Universal POS Tags (UPOS)

```
NOUN, VERB, ADJ, ADV, PRON, DET, ADP, AUX, CONJ, SCONJ,
PART, NUM, INTJ, PUNCT, SYM, X (other)
```

### Approach 1: HMM + Viterbi

Model tags as a Markov chain: $P(t_1, \ldots, t_n | w_1, \ldots, w_n)$

**Decompose** (with independence assumptions):

$$P(t_{1:n}, w_{1:n}) = \prod_{i=1}^n \underbrace{P(w_i | t_i)}_{\text{emission}} \cdot \underbrace{P(t_i | t_{i-1})}_{\text{transition}}$$

**Viterbi algorithm**: find most likely tag sequence in $O(nT^2)$ via dynamic programming:

$$\delta_t(j) = \max_{i} \delta_{t-1}(i) \cdot P(t_j | t_i) \cdot P(w_t | t_j)$$

### Approach 2: CRF (Conditional Random Field)

CRF jointly models the entire sequence:

$$P(t_{1:n} | w_{1:n}) = \frac{1}{Z} \exp\left(\sum_{i} \sum_k \lambda_k f_k(t_i, t_{i-1}, w_{1:n})\right)$$

CRF directly optimizes label sequence given observations stronger than HMM because features can look at the whole sentence.

### Approach 3: BERT-based (State of the Art)

Fine-tune BERT with a linear classification head over each token:
$$y_i = \text{softmax}(W h_i + b)$$

Achieves ~97-98% accuracy on Penn Treebank (vs HMM ~95%, CRF ~96%).

In [2]:
# Approach 1: HMM POS Tagger from scratch
from collections import defaultdict, Counter
import numpy as np

class HMMPosTagger:
    def __init__(self):
        self.transition = defaultdict(Counter)  # P(tag_i | tag_{i-1})
        self.emission = defaultdict(Counter)    # P(word | tag)
        self.initial = Counter()                # P(first tag)
        self.tags = set()
        
    def train(self, sentences):
        for sentence in sentences:
            words, tags = zip(*sentence)
            self.initial[tags[0]] += 1
            for i, (w, t) in enumerate(sentence):
                self.emission[t][w] += 1
                self.tags.add(t)
                if i > 0:
                    self.transition[tags[i-1]][t] += 1
                    
    def viterbi(self, words):
        tags = list(self.tags)
        T, N = len(words), len(tags)
        
        # Initialize
        viterbi = np.full((T, N), -np.inf)
        backpointer = np.zeros((T, N), dtype=int)
        
        init_total = sum(self.initial.values())
        for j, tag in enumerate(tags):
            init_prob = (self.initial[tag] + 1) / (init_total + len(tags))  # Laplace
            emit_prob = (self.emission[tag][words[0]] + 1) / (sum(self.emission[tag].values()) + 10000)
            viterbi[0, j] = np.log(init_prob) + np.log(emit_prob)
        
        # Fill
        for t in range(1, T):
            for j, curr_tag in enumerate(tags):
                trans_total = {tag: sum(self.transition[tag].values()) for tag in tags}
                emit_total = sum(self.emission[curr_tag].values())
                
                scores = []
                for i, prev_tag in enumerate(tags):
                    trans = (self.transition[prev_tag][curr_tag] + 1) / (trans_total.get(prev_tag, 0) + len(tags))
                    scores.append(viterbi[t-1, i] + np.log(trans))
                
                emit = (self.emission[curr_tag][words[t]] + 1) / (emit_total + 10000)
                best = np.argmax(scores)
                viterbi[t, j] = scores[best] + np.log(emit)
                backpointer[t, j] = best
        
        # Backtrack
        result = []
        last_tag = int(np.argmax(viterbi[T-1]))
        result.append(tags[last_tag])
        for t in range(T-1, 0, -1):
            last_tag = backpointer[t, last_tag]
            result.append(tags[last_tag])
        return list(zip(words, reversed(result)))

# Toy training data
train_data = [
    [("the", "DT"), ("dog", "NN"), ("runs", "VBZ"), ("fast", "RB")],
    [("a", "DT"), ("cat", "NN"), ("jumps", "VBZ"), ("high", "RB")],
    [("the", "DT"), ("bird", "NN"), ("sings", "VBZ"), ("loudly", "RB")],
    [("dogs", "NNS"), ("run", "VBP"), ("quickly", "RB")],
]

tagger = HMMPosTagger()
tagger.train(train_data)

test = ["the", "cat", "runs", "fast"]
result = tagger.viterbi(test)
print("HMM Viterbi POS tagging:")
for word, tag in result:
    print(f"  {word:15s} → {tag}")

HMM Viterbi POS tagging:
  the             → DT
  cat             → NN
  runs            → VBZ
  fast            → RB


In [3]:
# spaCy POS Tagging (production)
import spacy

nlp = spacy.load('en_core_web_sm')

text = "Apple is looking at buying a U.K. startup for $1 billion"
doc = nlp(text)

print(f"{'Token':<12} {'POS':>6} {'Fine-grained':>12} {'Dep':>8}")
print("-" * 44)
for token in doc:
    print(f"{token.text:<12} {token.pos_:>6} {token.tag_:>12} {token.dep_:>8}")

# BERT-based POS with HuggingFace
from transformers import pipeline

pos_pipeline = pipeline("token-classification", model="vblagoje/bert-english-uncased-finetuned-pos",
                         aggregation_strategy=None)
result = pos_pipeline(text)

print("\nBERT POS tagger:")
for item in result:
    print(f"  {item['word']:<15} {item['entity']:<8} ({item['score']:.3f})")

Token           POS Fine-grained      Dep
--------------------------------------------
Apple         PROPN          NNP    nsubj
is              AUX          VBZ      aux
looking        VERB          VBG     ROOT
at              ADP           IN     prep
buying         VERB          VBG    pcomp
a               DET           DT      det
U.K.          PROPN          NNP     dobj
startup        NOUN           NN    advcl
for             ADP           IN     prep
$               SYM            $ quantmod
1               NUM           CD compound
billion         NUM           CD     pobj


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: vblagoje/bert-english-uncased-finetuned-pos
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BERT POS tagger:
  apple           PROPN    (0.999)
  is              AUX      (0.997)
  looking         VERB     (1.000)
  at              SCONJ    (0.998)
  buying          VERB     (1.000)
  a               DET      (0.999)
  u               PROPN    (0.996)
  .               PUNCT    (1.000)
  k               PROPN    (0.995)
  .               PUNCT    (1.000)
  startup         NOUN     (0.999)
  for             ADP      (0.999)
  $               SYM      (0.997)
  1               NUM      (0.999)
  billion         NUM      (0.998)


## 2. Dependency Parsing

Dependency parsing identifies **grammatical dependencies** between words, producing a directed tree.

### Universal Dependencies (UD)

```
"Apple bought the startup"

ROOT
 └── bought (VERB)
      ├── Apple (nsubj) ← nominal subject
      └── startup (obj) ← direct object
           └── the (det) ← determiner
```

Common UD relations: `nsubj`, `obj`, `iobj`, `csubj`, `det`, `nmod`, `amod`, `advmod`, `case`, `conj`, `cc`, `root`

### Arc-Standard (Transition-Based)

Uses a **stack** $\sigma$ and **buffer** $\beta$. Three transitions:

| Transition | Action |
|-----------|--------|
| **SHIFT** | Move $\beta[0]$ onto $\sigma$ |
| **LEFT-ARC(r)** | Add arc $\sigma[0] \leftarrow \sigma[1]$ with relation $r$, pop $\sigma[1]$ |
| **RIGHT-ARC(r)** | Add arc $\sigma[0] \rightarrow \sigma[1]$ with relation $r$, pop $\sigma[0]$ |

The parser is a **classifier** over stack/buffer features (LSTM/BERT) that predicts the next transition.

### Arc-Eager

Adds **REDUCE** transition (pop stack without creating arc). Creates right-arc attachments earlier more efficient for long-range dependencies.

### Graph-Based (Eisner Algorithm)

Scores all possible arcs $s(i \rightarrow j)$ and finds the **maximum spanning tree** (MST):

$$T^* = \arg\max_{T \in \mathcal{T}(n)} \sum_{(i,j) \in T} s(i \rightarrow j)$$

Eisner's algorithm finds the optimal **projective** MST in $O(n^3)$. Handles long-range dependencies better than transition-based approaches.

In [4]:
import spacy
from spacy import displacy

nlp = spacy.load('en_core_web_sm')

# Dependency parsing
texts = [
    "The quick brown fox jumps over the lazy dog",
    "Apple acquired the startup for one billion dollars"
]

for text in texts:
    doc = nlp(text)
    print(f"\nText: '{text}'")
    print(f"{'Token':<12} {'Head':<12} {'Dep':<12} {'Children'}")
    print("-" * 55)
    for token in doc:
        children = [c.text for c in token.children]
        print(f"{token.text:<12} {token.head.text:<12} {token.dep_:<12} {children}")

# Traverse dependency tree
def find_subject_verb_object(doc):
    """Extract SVO triples from dependency parse."""
    triples = []
    for token in doc:
        if token.dep_ in ('nsubj', 'nsubjpass'):
            verb = token.head
            obj = [c for c in verb.children if c.dep_ in ('obj', 'dobj', 'pobj')]
            if obj:
                triples.append((token.text, verb.text, obj[0].text))
    return triples

doc = nlp("Apple acquired the startup. The company paid one billion dollars.")
triples = find_subject_verb_object(doc)
print("\nSVO Triples:")
for s, v, o in triples:
    print(f"  ({s}, {v}, {o})")


Text: 'The quick brown fox jumps over the lazy dog'
Token        Head         Dep          Children
-------------------------------------------------------
The          fox          det          []
quick        fox          amod         []
brown        fox          amod         []
fox          jumps        nsubj        ['The', 'quick', 'brown']
jumps        jumps        ROOT         ['fox', 'over']
over         jumps        prep         ['dog']
the          dog          det          []
lazy         dog          amod         []
dog          over         pobj         ['the', 'lazy']

Text: 'Apple acquired the startup for one billion dollars'
Token        Head         Dep          Children
-------------------------------------------------------
Apple        acquired     nsubj        []
acquired     acquired     ROOT         ['Apple', 'startup', 'for']
the          startup      det          []
startup      acquired     dobj         ['the']
for          acquired     prep         ['dollars'

## 3. Constituency Parsing

Constituency parsing produces **phrase-structure trees** that show how words group into phrases.

```
"The cat sat on the mat"

S
├── NP (The cat)
│   ├── DT (The)
│   └── NN (cat)
└── VP (sat on the mat)
    ├── VBD (sat)
    └── PP (on the mat)
        ├── IN (on)
        └── NP (the mat)
            ├── DT (the)
            └── NN (mat)
```

### CKY (Cocke-Kasami-Younger) Algorithm

Dynamic programming over **binarized CFG** (CNF Chomsky Normal Form):

$$\text{score}[i][j][X] = \max_{i \leq k < j, X \rightarrow Y Z} \text{score}[i][k][Y] \cdot \text{score}[k+1][j][Z] \cdot P(X \rightarrow YZ)$$

Time: $O(n^3 |G|)$ where $|G|$ is grammar size. Parses from bottom up, filling a triangular chart.

### Neural Constituency Parsing

Modern approaches:
1. **Chart parsers**: Score all possible spans $(i, j, X)$ with BERT
2. **Span-based**: Learn span representations, predict label for each span
3. **Seq2seq**: Generate linearized trees with T5/GPT

In [5]:
# CKY algorithm
from typing import Dict, List, Tuple, Optional

class CKYParser:
    def __init__(self, grammar: Dict, lexicon: Dict):
        """grammar: {A: [(B,C,prob)...]}, lexicon: {word: [(tag,prob)...]}"""
        self.grammar = grammar
        self.lexicon = lexicon
        
    def parse(self, words: List[str]) -> Optional[Tuple]:
        n = len(words)
        # chart[i][j][symbol] = (prob, backpointer)
        chart = [[defaultdict(lambda: (0, None)) for _ in range(n)] for _ in range(n)]
        
        # Initialize with lexical rules
        for i, word in enumerate(words):
            for tag, prob in self.lexicon.get(word, [("UNK", 0.001)]):
                chart[i][i][tag] = (prob, word)
                
        # Fill chart (span length 2 to n)
        for span in range(2, n+1):
            for start in range(n - span + 1):
                end = start + span - 1
                for split in range(start, end):
                    for lhs, rules in self.grammar.items():
                        for (B, C, prob) in rules:
                            b_prob = chart[start][split].get(B, (0, None))[0]
                            c_prob = chart[split+1][end].get(C, (0, None))[0]
                            if b_prob > 0 and c_prob > 0:
                                new_prob = prob * b_prob * c_prob
                                if new_prob > chart[start][end][lhs][0]:
                                    chart[start][end][lhs] = (new_prob, (split, B, C))
        
        return chart[0][n-1].get("S", None)


# Simple toy grammar (Chomsky Normal Form)
grammar = {
    "S":  [("NP", "VP", 1.0)],
    "VP": [("VBD", "NP", 0.7), ("VBD", "PP", 0.3)],
    "NP": [("DT", "NN", 0.8), ("DT", "NNS", 0.2)],
    "PP": [("IN", "NP", 1.0)],
}
lexicon = {
    "the": [("DT", 1.0)],
    "cat": [("NN", 1.0)],
    "sat": [("VBD", 1.0)],
    "on":  [("IN", 1.0)],
    "mat": [("NN", 1.0)],
}

parser = CKYParser(grammar, lexicon)
sentence = ["the", "cat", "sat", "on", "the", "mat"]
result = parser.parse(sentence)
print(f"CKY parse of '{' '.join(sentence)}':")
if result:
    print(f"  Found parse with prob: {result[0]:.6f}")
else:
    print("  No parse found")

# Production: Berkeley Neural Parser
print("\nProduction constituency parsing (benepar):")
print("  pip install benepar")
print("  import benepar")
print("  benepar.download('benepar_en3')")
print("  nlp.add_pipe('benepar', config={'model': 'benepar_en3'})")
print("  doc = nlp(text)")
print("  print(list(doc.sents)[0]._.parse_string)")

CKY parse of 'the cat sat on the mat':
  Found parse with prob: 0.192000

Production constituency parsing (benepar):
  pip install benepar
  import benepar
  benepar.download('benepar_en3')
  nlp.add_pipe('benepar', config={'model': 'benepar_en3'})
  doc = nlp(text)
  print(list(doc.sents)[0]._.parse_string)


## 4. Coreference Resolution

Coreference resolution identifies all **mentions** in a text that refer to the same real-world entity.

**Example**:
> "**Apple** announced a new product. **The company** said **it** would ship in March."

Coreference chains: {"Apple", "The company", "it"} → same entity.

### Pipeline

1. **Mention detection**: find all spans that could be entity mentions (noun phrases, pronouns)
2. **Mention scoring**: compute pairwise scores between candidate mentions
3. **Clustering**: link mentions that are coreferent

### SpanBERT (2020)

SpanBERT extends BERT with:
- **Span masking**: mask contiguous spans (not random tokens)
- **SBO** (Span Boundary Objective): predict masked tokens using span boundary representations

For coreference: represent each mention as $g_i = [\text{BERT}(start_i); \text{BERT}(end_i); \hat{x}_i]$ where $\hat{x}_i$ is an attention-weighted span embedding.

Scoring: $$s(i, j) = s_m(i) + s_m(j) + s_c(i, j)$$

where $s_m$ = mention score, $s_c$ = compatibility score.

### Modern Approaches

- **f-coref** (2021): Fast coreference with lightweight cross-encoders
- **LingMess** (2022): Linguistically motivated mention scoring
- **LLM-based**: Prompt GPT-4 to resolve coreferences directly

In [6]:
# Coreference Resolution with f-coref (fast-coref)
# pip install fastcoref spacy
# python -m spacy download en_core_web_sm

def demo_fastcoref():
    from fastcoref import FCoref
    
    model = FCoref(device='cpu')  # or 'cuda'
    
    texts = [
        "Apple announced a new iPhone. The company said it will ship in September.",
        "John told Mary that he would help her with the project. She was grateful."
    ]
    
    preds = model.predict(texts=texts)
    
    for text, pred in zip(texts, preds):
        print(f"Text: {text}")
        for cluster in pred.get_clusters():
            mentions = [text[start:end] for start, end in cluster]
            print(f"  Cluster: {mentions}")
        print()

# Neural coreference from scratch (simplified mention-pair)
import torch
import torch.nn as nn

class SimpleMentionPairModel(nn.Module):
    def __init__(self, d_model=768, hidden=256):
        super().__init__()
        # Features: [span_i, span_j, span_i * span_j, |span_i - span_j|]
        self.scorer = nn.Sequential(
            nn.Linear(4 * d_model, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1)
        )
    
    def forward(self, span_i, span_j):
        features = torch.cat([
            span_i, span_j,
            span_i * span_j,
            torch.abs(span_i - span_j)
        ], dim=-1)
        return self.scorer(features)

model = SimpleMentionPairModel()
print(f"Mention-pair scorer: {sum(p.numel() for p in model.parameters()):,} params")

# Simulate
n_mentions = 10
span_embeds = torch.randn(n_mentions, 768)
scores = torch.zeros(n_mentions, n_mentions)
for i in range(n_mentions):
    for j in range(i):
        score = model(span_embeds[i:i+1], span_embeds[j:j+1])
        scores[i, j] = scores[j, i] = score.item()

print(f"Mention pair score matrix: {scores.shape}")
print(f"Highest scoring pair: ({scores.triu(1).argmax().item() // n_mentions},"
      f" {scores.triu(1).argmax().item() % n_mentions})")

Mention-pair scorer: 786,945 params
Mention pair score matrix: torch.Size([10, 10])
Highest scoring pair: (3, 7)


## 5. Relation Extraction

Given entities in text, extract **semantic relations** between them.

**Example**:
> "[Steve Jobs] founded [Apple] in [1976]"
> Relations: (Steve Jobs, founded, Apple), (Apple, founded_year, 1976)

### Pipeline vs Joint Extraction

- **Pipeline**: NER → RE (two separate models). Simpler but error propagation.
- **Joint**: Simultaneously extract entities and relations. More complex but better.

### BERT-based RE

Mark entities with special tokens, classify the pair:
```
Input: "[E1] Apple [/E1] was founded by [E2] Steve Jobs [/E2]"
Feature: [CLS] embedding → linear classifier → relation type
```

Or use **entity marker embeddings**: embed entity positions as features.

### TPLinker (2020): Table-Filling Joint Extraction

Models relation extraction as **table filling**:

$$T_{ij}^{r, \text{HE}} = 1 \text{ if token } i \text{ is the head-entity start for relation } r$$
$$T_{ij}^{r, \text{TE}} = 1 \text{ if token } j \text{ is the tail-entity start for relation } r$$
$$T_{ij}^{r, \text{EH-ET}} = 1 \text{ if } (i, j) \text{ links entity head } i \text{ to entity tail } j \text{ for relation } r$$

One forward pass produces all entity-relation triples simultaneously.

### REBEL (2021)

Seq2seq relation extraction: fine-tune BART to generate structured output:
```
Input:  "Steve Jobs co-founded Apple in 1976"
Output: "<triplet> Steve Jobs <subj> Apple <obj> founded <rel>"
```

In [7]:
# REBEL: seq2seq relation extraction
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def extract_relations_rebel(text, model, tokenizer, max_length=512):
    inputs = tokenizer(text, return_tensors='pt', max_length=max_length, truncation=True)
    output = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        length_penalty=0,
        num_beams=10,
        num_return_sequences=1,
    )
    decoded = tokenizer.batch_decode(output, skip_special_tokens=False)[0]
    return parse_rebel_output(decoded)

def parse_rebel_output(text):
    """Parse REBEL's structured output into (subject, relation, object) triples."""
    triples = []
    text = text.replace('<s>', '').replace('</s>', '').strip()
    parts = text.split('<triplet>')
    for part in parts[1:]:
        try:
            subj = part.split('<subj>')[0].strip()
            obj = part.split('<subj>')[1].split('<obj>')[0].strip()
            rel = part.split('<obj>')[1].split('<triplet>')[0].strip()
            if subj and obj and rel:
                triples.append((subj, rel, obj))
        except IndexError:
            continue
    return triples

# Load REBEL (uncomment to run)
# tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
# model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
# 
# text = "Steve Jobs co-founded Apple in 1976 in Cupertino, California."
# triples = extract_relations_rebel(text, model, tokenizer)
# for s, r, o in triples:
#     print(f"  ({s}, {r}, {o})")

# BERT entity-marker RE
class RelationExtractor(nn.Module):
    def __init__(self, bert_model, num_relations=10, d_model=768):
        super().__init__()
        self.bert = bert_model
        # Use [CLS] + entity marker representations
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 3, 256),  # [CLS], e1, e2
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_relations)
        )
    
    def forward(self, input_ids, attention_mask, e1_pos, e2_pos):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_output = outputs.last_hidden_state  # (B, T, d)
        
        cls_emb = seq_output[:, 0, :]                           # [CLS] token
        e1_emb  = seq_output[torch.arange(len(e1_pos)), e1_pos] # entity 1 start
        e2_emb  = seq_output[torch.arange(len(e2_pos)), e2_pos] # entity 2 start
        
        features = torch.cat([cls_emb, e1_emb, e2_emb], dim=-1)
        return self.classifier(features)

print("Relation extraction models loaded!")
print("REBEL: seq2seq extraction → directly generates (s, r, o) triples")
print("BERT-RE: entity-marker → classify relation type per entity pair")

Relation extraction models loaded!
REBEL: seq2seq extraction → directly generates (s, r, o) triples
BERT-RE: entity-marker → classify relation type per entity pair


## 6. Event Extraction

Event extraction identifies **events** (things that happen) and their **arguments** (who, what, where, when, how).

**ACE 2005 Schema Example**:
> "[Apple] acquired [Beats] for [$3 billion] in [2014]"
> - Event trigger: "acquired" (type: TRANSACTION/TRANSFER-OWNERSHIP)
> - Buyer: Apple
> - Artifact: Beats  
> - Price: $3 billion
> - Time: 2014

### Pipeline

1. **Trigger detection**: find event-triggering words/spans + classify event type
2. **Argument extraction**: for each trigger, find arguments and their roles

### DYGIE++ (2019)

Multi-task framework: NER + RE + Event Extraction with shared span representations:
- Enumerate all candidate spans
- Score each span for entity type (NER)
- Score span pairs for relation type (RE)
- Score spans as event triggers and argument roles (EE)
- Shared BERT encoder across all tasks

### OneIE (2020)

**Global joint extraction** models constraints across all IE tasks simultaneously:

$$\text{score}(y) = \sum_{\text{local}} s(y_i) + \sum_{\text{global}} s(y_i, y_j)$$

Uses a beam search over joint outputs to find globally consistent extractions.

In [8]:
# Event extraction with HuggingFace (ACE 2005 schema)
from transformers import pipeline

# Using a pre-trained event extraction model
# Available models: few dedicated ones; most use DYGIE++/OneIE frameworks

# Simplified event extraction with keyword spotting + argument detection
ACE_TRIGGER_VERBS = {
    "TRANSFER-OWNERSHIP": ["acquire", "buy", "purchase", "sell", "transfer"],
    "BORN": ["born", "gave birth"],
    "DIE": ["died", "killed", "passed away"],
    "MEET": ["met", "meeting", "conference", "summit"],
    "START-POSITION": ["appointed", "hired", "joined", "became CEO"],
    "END-POSITION": ["fired", "resigned", "left", "retired"],
}

def simple_event_detector(text, nlp):
    doc = nlp(text)
    events = []
    for token in doc:
        lemma = token.lemma_.lower()
        for event_type, triggers in ACE_TRIGGER_VERBS.items():
            if lemma in triggers:
                # Find arguments via dependency
                args = {"trigger": token.text, "event_type": event_type}
                for child in token.children:
                    if child.dep_ in ('nsubj', 'nsubjpass'):
                        args['agent'] = child.text
                    elif child.dep_ in ('obj', 'dobj'):
                        args['patient'] = child.text
                for prep in token.children:
                    if prep.dep_ == 'prep':
                        for obj in prep.children:
                            args[f'arg_{prep.text}'] = obj.text
                events.append(args)
    return events

import spacy
nlp = spacy.load('en_core_web_sm')

text = "Microsoft acquired Activision for $68.7 billion in 2023."
events = simple_event_detector(text, nlp)
print(f"Text: {text}")
print("Events:")
for e in events:
    print(f"  {e}")

print("\n--- DYGIE++ Framework ---")
print("allennlp predict dygie++ model.tar.gz data.jsonl --output-file output.jsonl")
print("Supports: NER + RE + Event Extraction jointly")

Text: Microsoft acquired Activision for $68.7 billion in 2023.
Events:
  {'trigger': 'acquired', 'event_type': 'TRANSFER-OWNERSHIP', 'agent': 'Microsoft', 'patient': 'Activision', 'arg_for': 'billion', 'arg_in': '2023'}

--- DYGIE++ Framework ---
allennlp predict dygie++ model.tar.gz data.jsonl --output-file output.jsonl
Supports: NER + RE + Event Extraction jointly


## 7. Text Style Transfer

Style transfer modifies the **style** of text (formality, sentiment, author voice, tense) while preserving the **content**.

### Key Challenge
Decompose: **content** vs **style**:
$$x = \underbrace{c}_{\text{content}} + \underbrace{s}_{\text{style}} \quad \longrightarrow \quad x' = c + s'$$

### Methods

**STRAP (2021)**: "Style Transfer via Authorship Preservation"
- Fine-tune author classifiers + use PPLM-style decoding
- Target: transfer to author's writing style

**PPLM (2019)**: Plug and Play Language Model
- Steer GPT-2 generation using gradients from a **discriminator**:
$$\Delta h_t \propto \nabla_{h_t} \log p(a | x)$$
- Update hidden states toward desired attribute $a$ (topic, sentiment)

**FUDGE (2021)**: Future Discriminators for Generation
- Train discriminator that predicts if **future** tokens will satisfy attribute $a$
$$P_\text{FUDGE}(x_t | x_{<t}) \propto P_\text{LM}(x_t | x_{<t}) \cdot P_d(a | x_{\leq t})$$

**ParaDiG (2022)**: Parallel Data-free Guided Text Generation with LLMs
- Use paraphrase + style classifier loss to guide generation

**Modern approach**: Prompt LLMs:
```
"Rewrite the following text in a formal style, preserving all information: {text}"
```

In [9]:
# Style Transfer with T5 (formal/informal)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Models for formality transfer
STYLE_MODELS = {
    "formality":   "prithivida/informal_to_formal_styletransfer",
    "detoxify":    "s-nlp/t5-paranmt-detox",
    "simplify":    "facebook/bart-large-cnn",  # for summarization/simplification
}

def style_transfer(text, model_name, prefix=""):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    input_text = prefix + text if prefix else text
    inputs = tokenizer(input_text, return_tensors='pt', max_length=512, truncation=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        temperature=1.0,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# PPLM-style gradient-based steering
class PPLMSteering:
    def __init__(self, lm, discriminator, step_size=0.01, num_iterations=3):
        self.lm = lm
        self.discriminator = discriminator
        self.step_size = step_size
        self.num_iterations = num_iterations
    
    def generate_with_attribute(self, input_ids, attribute):
        past = None
        generated = list(input_ids[0])
        
        for _ in range(50):  # max generation length
            with torch.enable_grad():
                outputs = self.lm(input_ids=input_ids, past_key_values=past, 
                                  use_cache=True, output_hidden_states=True)
                h_T = outputs.hidden_states[-1][:, -1, :].requires_grad_(True)
                
                # Gradient from discriminator
                attr_score = self.discriminator(h_T, attribute)
                grads = torch.autograd.grad(attr_score, h_T)[0]
                
                # Update hidden state
                h_T_updated = h_T + self.step_size * grads
            
            # Use updated hidden state to get next token
            logits = self.lm.lm_head(h_T_updated)
            next_token = logits.argmax(dim=-1)
            generated.append(next_token.item())
        
        return generated

print("Style transfer methods:")
examples = [
    ("Informal", "wanna grab lunch?"),
    ("Formal", "Would you like to join me for lunch?"),
    ("Negative", "This movie was terrible and boring."),
    ("Positive", "This movie was entertaining and engaging."),
]
for style, text in examples:
    print(f"  [{style}] {text}")

Style transfer methods:
  [Informal] wanna grab lunch?
  [Formal] Would you like to join me for lunch?
  [Negative] This movie was terrible and boring.
  [Positive] This movie was entertaining and engaging.


## 8. Text Simplification

Text simplification rewrites complex text into simpler, more accessible language while preserving meaning.

Applications: making medical/legal text accessible, helping language learners, NLP preprocessing.

### Simplification Operations
1. **Lexical**: replace complex words with simpler synonyms
2. **Syntactic**: split long sentences, remove passive voice, convert complex to simple clauses
3. **Content**: remove or compress less important information

### ACCESS (2020)
Controllable simplification with 4 simplicity attributes:
- **LEVAL** (length ratio): `<LEVAL_0.8>` shorten by 20%
- **LEVSIM** (character Levenshtein): preserve/modify words
- **WORDRK** (word rank): use simpler vocabulary
- **DEPTP** (dependency tree depth): simplify syntax

Format: `<LEVAL_0.8> <LEVSIM_0.75> <WORDRK_0.8> <DEPTP_0.4> The text to simplify`

### MUSS (Multilingual Unsupervised Sentence Simplification, 2022)
Uses paraphrase mining across complexity levels to create **pseudo-parallel data** without human annotation:
1. Mine paraphrases between Wikipedia and Simple Wikipedia
2. Filter by complexity difference
3. Fine-tune mBART on the pairs

In [10]:
# Text Simplification
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def access_simplify(text, leval=0.8, levsim=0.75, wordrk=0.8, deptp=0.4):
    """ACCESS: add control tokens before text."""
    prefix = f"LEVAL_{leval:.2f} LEVSIM_{levsim:.2f} WORDRK_{wordrk:.2f} DEPTP_{deptp:.2f} "
    return prefix + text

# T5-based simplification
def t5_simplify(text, model_name="t5-base"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    # T5 uses "simplify: " prefix for simplification
    input_text = f"simplify: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        no_repeat_ngram_size=2,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Readability metrics
def flesch_reading_ease(text):
    """Flesch Reading Ease score (higher = easier)."""
    sentences = text.split('.')
    words = text.split()
    syllables = sum(count_syllables(w) for w in words)
    
    if len(sentences) == 0 or len(words) == 0:
        return 0
    
    words_per_sentence = len(words) / len(sentences)
    syllables_per_word = syllables / len(words)
    
    return 206.835 - 1.015 * words_per_sentence - 84.6 * syllables_per_word

def count_syllables(word):
    word = word.lower().rstrip('.,!?')
    count = len([c for c in word if c in 'aeiouy'])
    return max(1, count)

complex_text = "The utilization of sophisticated lexical constructs in conjunction with \
intricate syntactic arrangements necessitates a comprehensive understanding of \
advanced linguistic principles."

simple_text = "Using complex words and long sentences requires knowing advanced grammar."

complex_score = flesch_reading_ease(complex_text)
simple_score = flesch_reading_ease(simple_text)

print(f"Complex text readability: {complex_score:.1f}  (Flesch, higher=easier)")
print(f"Simple text readability:  {simple_score:.1f}")

print(f"\nACCESS format: {access_simplify(complex_text[:50]+'...')}")

Complex text readability: -61.3  (Flesch, higher=easier)
Simple text readability:  24.1

ACCESS format: LEVAL_0.80 LEVSIM_0.75 WORDRK_0.80 DEPTP_0.40 The utilization of sophisticated lexical construct...


## 9. Keyword Extraction

Keyword extraction identifies the most representative words/phrases from a document.

### TF-IDF

$$\text{TF-IDF}(t, d) = \underbrace{\frac{f_{t,d}}{\sum_{t'} f_{t',d}}}_{\text{TF}} \times \underbrace{\log \frac{|D|}{|\{d \in D: t \in d\}|}}_{\text{IDF}}$$

High TF-IDF = frequent in this document, rare across corpus.

### RAKE (Rapid Automatic Keyword Extraction, 2010)

Uses word co-occurrence in phrases:
1. Split text on stopwords/punctuation → candidate phrases
2. Score words: $\text{score}(w) = \frac{\text{freq\_in\_phrases}(w)}{\text{phrase\_count}(w)}$
3. Score phrase = sum of word scores

### YAKE (2020)

Unsupervised, statistical features (no co-occurrence graph needed):
- Term frequency $TF_n(t)$ normalized
- Position: $WOffset$ early terms score higher
- Contextual diversity: $DL, DR$ left/right context variety
- Sentence-level: $TSentences$ appears in many sentences
- Co-occurrence with stopwords: $WRelStopWords$

$$S(t) = \frac{WL \cdot WR \cdot WRelStopWords}{TF_n(t) \cdot (\frac{1}{3}WL + WR + WRelStopWords + 1)}$$

### KeyBERT (2020)

Use BERT embeddings to find phrases most similar to the document:

1. Extract candidate phrases (n-grams)
2. Embed document → $\phi(\text{doc})$
3. Embed each candidate → $\phi(\text{phrase}_i)$
4. Rank by $\cos(\phi(\text{doc}), \phi(\text{phrase}_i))$
5. Diversify with MMR (Maximal Marginal Relevance)

In [11]:
# KeyBERT keyword extraction
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

# Also: YAKE, RAKE
# pip install keybert yake rake-nltk

text = """
Transformer models have revolutionized natural language processing. The attention mechanism
allows models to capture long-range dependencies in text. BERT, GPT, and T5 are among
the most influential transformer architectures. Fine-tuning pre-trained transformers on
downstream tasks has become the standard approach in NLP. Parameter-efficient methods
like LoRA enable fine-tuning large language models with limited computational resources.
"""

# TF-IDF baseline
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def tfidf_keywords(text, top_k=10):
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    tfidf = vectorizer.fit_transform([text])
    scores = tfidf.toarray()[0]
    vocab = vectorizer.get_feature_names_out()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(vocab[i], scores[i]) for i in top_idx]

print("TF-IDF Keywords:")
for kw, score in tfidf_keywords(text, top_k=8):
    print(f"  [{score:.4f}] {kw}")

# KeyBERT (semantic)
kw_model = KeyBERT(model='all-MiniLM-L6-v2')

print("\nKeyBERT Keywords (semantic similarity):")
keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1, 2),
    stop_words='english',
    top_n=8,
    use_mmr=True,    # Maximal Marginal Relevance for diversity
    diversity=0.5
)
for kw, score in keywords:
    print(f"  [{score:.4f}] {kw}")

# YAKE
print("\nYAKE Keywords (statistical):")
import yake
yake_extractor = yake.KeywordExtractor(lan="en", n=2, dedupLim=0.7, top=8)
yake_kws = yake_extractor.extract_keywords(text)
for kw, score in yake_kws:
    print(f"  [{score:.4f}] {kw}")  # lower score = more important in YAKE

TF-IDF Keywords:
  [0.2928] models
  [0.1952] transformer
  [0.1952] language
  [0.1952] fine tuning
  [0.1952] fine
  [0.1952] tuning
  [0.0976] tuning large
  [0.0976] tuning pre


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


KeyBERT Keywords (semantic similarity):
  [0.4739] language models
  [0.4666] nlp parameter
  [0.4217] processing attention
  [0.3828] bert gpt
  [0.3745] transformers downstream
  [0.2721] range dependencies
  [0.2462] tuning pre
  [0.1923] t5 influential

YAKE Keywords (statistical):


  [0.0673] revolutionized natural
  [0.1241] language processing
  [0.1767] processing
  [0.1857] models
  [0.1860] natural language
  [0.2111] BERT
  [0.2111] GPT
  [0.2512] revolutionized


## 10. Text Similarity: BM25 vs Dense Retrieval

### BM25 (Best Match 25)

Probabilistic sparse retrieval improved TF-IDF:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{TF}(t,d) \cdot (k_1 + 1)}{\text{TF}(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

Where:
- $k_1 \in [1.2, 2.0]$ term frequency saturation
- $b = 0.75$ length normalization
- $\text{avgdl}$ average document length

**Strengths**: Exact lexical match, no training needed, fast, explainable.
**Weaknesses**: Vocabulary mismatch ("car" vs "automobile" = 0 overlap).

### Dense Retrieval (DPR, 2020)

Encode query and document separately:
$$\text{sim}(q, d) = \phi_q(q)^T \phi_d(d)$$

Train with in-batch negatives + hard negatives.

**Strengths**: Semantic matching, handles synonyms.
**Weaknesses**: Requires training, misses exact matches, larger index.

### Hybrid Retrieval

Best of both worlds:
$$\text{score}(q, d) = \alpha \cdot \text{BM25}(q, d) + (1-\alpha) \cdot \text{dense}(q, d)$$

In [12]:
# BM25 from scratch
import numpy as np
from collections import Counter
import math

class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus = []
        self.doc_freqs = []
        self.idf = {}
        self.avgdl = 0
        
    def fit(self, corpus):
        self.corpus = [doc.lower().split() for doc in corpus]
        self.N = len(self.corpus)
        self.avgdl = sum(len(d) for d in self.corpus) / self.N
        self.doc_freqs = [Counter(doc) for doc in self.corpus]
        
        # IDF
        df = Counter()
        for doc in self.corpus:
            for term in set(doc):
                df[term] += 1
        self.idf = {term: math.log((self.N - df[term] + 0.5) / (df[term] + 0.5) + 1)
                    for term in df}
    
    def score(self, query, doc_idx):
        query_terms = query.lower().split()
        doc = self.corpus[doc_idx]
        doc_freq = self.doc_freqs[doc_idx]
        score = 0
        for term in query_terms:
            if term not in self.idf:
                continue
            tf = doc_freq.get(term, 0)
            idf = self.idf[term]
            norm = tf * (self.k1 + 1)
            denom = tf + self.k1 * (1 - self.b + self.b * len(doc) / self.avgdl)
            score += idf * norm / denom
        return score
    
    def retrieve(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: -x[1])[:top_k]

# Test
corpus = [
    "deep learning neural networks machine learning",
    "transformer attention mechanism BERT NLP",
    "Python programming software engineering",
    "natural language processing text classification",
    "convolutional neural network image recognition",
]
bm25 = BM25()
bm25.fit(corpus)

query = "neural network for language"
results = bm25.retrieve(query, top_k=3)

print(f"BM25 retrieval for: '{query}'")
for idx, score in results:
    print(f"  [{score:.3f}] {corpus[idx]}")

# Dense retrieval with sentence-transformers
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = model.encode(corpus, convert_to_tensor=True)
query_embedding = model.encode(query, convert_to_tensor=True)
dense_scores = util.cos_sim(query_embedding, doc_embeddings)[0]

print(f"\nDense retrieval for: '{query}'")
top3 = dense_scores.topk(3)
for score, idx in zip(top3.values, top3.indices):
    print(f"  [{score:.3f}] {corpus[idx]}")

BM25 retrieval for: 'neural network for language'
  [2.262] convolutional neural network image recognition
  [1.386] natural language processing text classification
  [0.803] deep learning neural networks machine learning


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Dense retrieval for: 'neural network for language'
  [0.424] natural language processing text classification
  [0.423] transformer attention mechanism BERT NLP
  [0.401] deep learning neural networks machine learning


## 11. Paraphrase Detection and Mining

### Paraphrase Detection

Binary classification: are two sentences paraphrases?

**PAWS Benchmark** (2019): Paraphrase Adversaries from Word Scrambling adversarial pairs that have high word overlap but different meanings.

Example:
> Paraphrase: "Flights from New York to Florida" ≈ "Flights to Florida from New York"
> Non-paraphrase: "Flights from New York to Florida" ≠ "Flights from Florida to New York"

**Model**: Siamese BERT or cross-encoder BERT fine-tuned on labeled pairs.

### Large-Scale Paraphrase Mining

Mine paraphrases from billions of sentence pairs:
1. Embed all sentences
2. Find nearest neighbors efficiently (FAISS)
3. Filter by similarity threshold and diversity

**SBERT Paraphrase Mining** (util.paraphrase_mining): optimized implementation for large corpora.

In [13]:
# Paraphrase detection and mining
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# Cross-encoder for accurate paraphrase detection
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/stsb-roberta-large')

pairs = [
    ("A man is eating food", "A man is eating a piece of bread"),   # paraphrase
    ("A man is eating food", "A man is playing guitar"),             # not paraphrase
    ("Flights from NY to FL", "Flights to FL from NY"),             # paraphrase
    ("Flights from NY to FL", "Flights from FL to NY"),             # NOT paraphrase!
]

scores = cross_encoder.predict(pairs)
print("Paraphrase Detection (cross-encoder):")
for (s1, s2), score in zip(pairs, scores):
    verdict = "PARAPHRASE" if score > 0.5 else "DIFFERENT"
    print(f"  [{score:.3f}] {verdict}")
    print(f"    S1: {s1}")
    print(f"    S2: {s2}")
    print()

# Large-scale mining
sentences = [
    "The weather is lovely today",
    "It's a beautiful day outside",
    "How do I train a neural network?",
    "What is the process of training deep learning models?",
    "Paris is the capital of France",
    "France's capital city is Paris",
    "I enjoy reading books",
    "Books are my favorite pastime",
]

model = SentenceTransformer('all-MiniLM-L6-v2')
paraphrase_pairs = util.paraphrase_mining(model, sentences)

print("Mined Paraphrase Pairs (threshold=0.7):")
for score, i, j in paraphrase_pairs:
    if score < 0.7:
        continue
    print(f"  [{score:.3f}] '{sentences[i]}' ↔ '{sentences[j]}'")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Paraphrase Detection (cross-encoder):
  [0.643] PARAPHRASE
    S1: A man is eating food
    S2: A man is eating a piece of bread

  [0.068] DIFFERENT
    S1: A man is eating food
    S2: A man is playing guitar

  [0.969] PARAPHRASE
    S1: Flights from NY to FL
    S2: Flights to FL from NY

  [0.969] PARAPHRASE
    S1: Flights from NY to FL
    S2: Flights from FL to NY



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Mined Paraphrase Pairs (threshold=0.7):
  [0.967] 'Paris is the capital of France' ↔ 'France's capital city is Paris'
  [0.768] 'The weather is lovely today' ↔ 'It's a beautiful day outside'


## 12. Document Understanding: LayoutLM, Donut, Nougat

Document understanding processes documents with **visual layout** (PDFs, scanned forms, invoices, receipts).

### LayoutLM (Microsoft, 2020)

Extends BERT with **2D position embeddings** from OCR bounding boxes:

$$\text{Embed}(w_i) = \text{WordEmbed}(w_i) + \text{PosEmbed}(x_1^i, y_1^i, x_2^i, y_2^i, h_i, w_i)$$

Pipeline: OCR → extract words + bounding boxes → feed to LayoutLM → classify/extract.

Versions: LayoutLM → LayoutLMv2 (visual features + cross-attention) → LayoutLMv3 (unified multimodal)

### Donut (2022)

**Document Understanding Transformer** no OCR needed!

Architecture: Vision Encoder (Swin Transformer) + Text Decoder (BART)
- Input: document image → CNN encoder
- Output: structured JSON/text via autoregressive decoding
- Task prompt: `<s_cord>` (receipt), `<s_docvqa>` (VQA), `<s_rvlcdip>` (classification)

### Nougat (Meta, 2023)

**Neural Optical Understanding for Academic Documents** parse scientific PDFs into Markdown:
- Architecture: Donut-style (encoder-decoder)
- Preserves: equations (LaTeX), tables, figures, references
- Training: pairs of (PDF page, structured Markdown)

### GOT-OCR2.0 (2024)

General OCR Theory unified OCR for any document type:
- Plain text, math equations, tables, charts, sheet music, geometric shapes
- Architecture: encoder-decoder with image patches

In [14]:
# LayoutLMv3 for document understanding
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image
import torch

def layoutlm_extract(image_path, model_name="microsoft/layoutlmv3-base"):
    processor = LayoutLMv3Processor.from_pretrained(model_name, apply_ocr=True)
    model = LayoutLMv3ForTokenClassification.from_pretrained(model_name)
    
    image = Image.open(image_path).convert("RGB")
    encoding = processor(image, return_tensors="pt", truncation=True)
    
    with torch.no_grad():
        outputs = model(**encoding)
    
    logits = outputs.logits
    predicted_ids = logits.argmax(-1).squeeze().tolist()
    predicted_labels = [model.config.id2label[idx] for idx in predicted_ids]
    return predicted_labels

# Donut for document VQA
def donut_vqa(image_path, question, model_name="naver-clova-ix/donut-base-finetuned-docvqa"):
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    
    processor = DonutProcessor.from_pretrained(model_name)
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values
    
    # Format prompt
    task_prompt = f"<s_docvqa><s_question>{question}</s_question><s_answer>"
    decoder_input_ids = processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids
    
    outputs = model.generate(
        pixel_values, decoder_input_ids=decoder_input_ids,
        max_length=model.decoder.config.max_position_embeddings
    )
    
    answer = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return answer

# Nougat for scientific PDFs
print("Nougat usage:")
print("  pip install nougat-ocr")
print("  nougat paper.pdf -o output_dir/")
print("  # Outputs: paper.mmd (structured Markdown with equations)")
print()

print("Document Understanding Models Summary:")
models = [
    ("LayoutLMv3",    "With OCR",  "NER, key-value extraction",   "OCR + bbox required"),
    ("Donut",         "OCR-free",  "VQA, classification, IE",      "End-to-end from image"),
    ("Nougat",        "OCR-free",  "Scientific PDF → Markdown",    "Academic papers"),
    ("GOT-OCR2.0",    "OCR-free",  "Any document type",            "Unified OCR"),
    ("TrOCR",         "OCR model", "Printed+handwritten OCR",      "Pure OCR"),
]
print(f"{'Model':<15} {'OCR':<12} {'Tasks':<35} {'Notes'}")
print("-" * 85)
for m, ocr, tasks, notes in models:
    print(f"{m:<15} {ocr:<12} {tasks:<35} {notes}")

Nougat usage:
  pip install nougat-ocr
  nougat paper.pdf -o output_dir/
  # Outputs: paper.mmd (structured Markdown with equations)

Document Understanding Models Summary:
Model           OCR          Tasks                               Notes
-------------------------------------------------------------------------------------
LayoutLMv3      With OCR     NER, key-value extraction           OCR + bbox required
Donut           OCR-free     VQA, classification, IE             End-to-end from image
Nougat          OCR-free     Scientific PDF → Markdown           Academic papers
GOT-OCR2.0      OCR-free     Any document type                   Unified OCR
TrOCR           OCR model    Printed+handwritten OCR             Pure OCR


## 13. Full Information Extraction Pipeline

In [15]:
# Complete IE pipeline: NER + RE + Event Extraction
import spacy
from transformers import pipeline

nlp = spacy.load('en_core_web_sm')

class IEPipeline:
    def __init__(self):
        self.ner = nlp
        # In production: use dedicated RE and EE models
        
    def run(self, text):
        doc = self.ner(text)
        
        # 1. Named Entity Recognition
        entities = [(ent.text, ent.label_, ent.start_char, ent.end_char) 
                    for ent in doc.ents]
        
        # 2. Dependency-based Relation Extraction
        relations = []
        for token in doc:
            if token.dep_ in ('nsubj', 'nsubjpass') and token.head.pos_ == 'VERB':
                subj = token
                verb = token.head
                for child in verb.children:
                    if child.dep_ in ('obj', 'dobj', 'pobj', 'attr'):
                        relations.append({
                            "subject": subj.text,
                            "predicate": verb.lemma_,
                            "object": child.text
                        })
        
        # 3. Simple Event Detection
        events = []
        event_verbs = {"acquire": "ACQUISITION", "found": "FOUNDING", 
                       "hire": "HIRING", "announce": "ANNOUNCEMENT",
                       "invest": "INVESTMENT", "partner": "PARTNERSHIP"}
        for token in doc:
            if token.lemma_ in event_verbs:
                event = {"type": event_verbs[token.lemma_], "trigger": token.text}
                for child in token.children:
                    if child.dep_ == 'nsubj': event['agent'] = child.text
                    elif child.dep_ in ('obj', 'dobj'): event['patient'] = child.text
                events.append(event)
        
        return {
            "text": text,
            "entities": entities,
            "relations": relations,
            "events": events
        }

# Test the pipeline
ie = IEPipeline()

texts = [
    "Microsoft acquired Activision Blizzard for $68.7 billion in 2023.",
    "Sam Altman founded OpenAI in San Francisco in 2015.",
    "Apple invested $1 billion in TSMC's new chip fabrication plant.",
]

for text in texts:
    result = ie.run(text)
    print(f"Text: {text}")
    print(f"  Entities:  {[(e[0], e[1]) for e in result['entities']]}")
    print(f"  Relations: {result['relations']}")
    print(f"  Events:    {result['events']}")
    print()

Text: Microsoft acquired Activision Blizzard for $68.7 billion in 2023.
  Entities:  [('Microsoft', 'ORG'), ('Activision Blizzard', 'PERSON'), ('$68.7 billion', 'MONEY'), ('2023', 'DATE')]
  Relations: [{'subject': 'Microsoft', 'predicate': 'acquire', 'object': 'Blizzard'}]
  Events:    [{'type': 'ACQUISITION', 'trigger': 'acquired', 'agent': 'Microsoft', 'patient': 'Blizzard'}]

Text: Sam Altman founded OpenAI in San Francisco in 2015.
  Entities:  [('Sam Altman', 'PERSON'), ('OpenAI', 'GPE'), ('San Francisco', 'GPE'), ('2015', 'DATE')]
  Relations: [{'subject': 'Altman', 'predicate': 'found', 'object': 'OpenAI'}]
  Events:    [{'type': 'FOUNDING', 'trigger': 'founded', 'agent': 'Altman', 'patient': 'OpenAI'}]

Text: Apple invested $1 billion in TSMC's new chip fabrication plant.
  Entities:  [('Apple', 'ORG'), ('$1 billion', 'MONEY'), ('TSMC', 'ORG')]
  Relations: [{'subject': 'Apple', 'predicate': 'invest', 'object': 'billion'}]
  Events:    [{'type': 'INVESTMENT', 'trigger': 'inves

## Additional Learning Resources

### Foundational Papers

| Topic | Paper | Link |
|-------|-------|------|
| POS Tagging (CRF) | "Conditional Random Fields" (Lafferty et al.) | https://arxiv.org/abs/1011.4088 |
| Dependency Parsing | "A Fast and Accurate Dependency Parser" (Chen & Manning) | https://aclanthology.org/D14-1082 |
| Constituency Parsing | "Constituency Parsing with a Self-Attentive Encoder" | https://arxiv.org/abs/1805.01052 |
| SpanBERT | "SpanBERT: Improving Pre-training by Representing Spans" | https://arxiv.org/abs/1907.10529 |
| f-coref | "Fast Coreference Resolution" | https://arxiv.org/abs/2109.04443 |
| TPLinker | "TPLinker: Single-stage Joint Extraction" | https://arxiv.org/abs/2010.13415 |
| REBEL | "REBEL: Relation Extraction By End-to-end Language generation" | https://arxiv.org/abs/2108.03141 |
| DYGIE++ | "Entity, Relation, and Event Extraction with Contextualized Span Representations" | https://arxiv.org/abs/1909.03645 |
| PPLM | "Plug and Play Language Models" | https://arxiv.org/abs/1912.02164 |
| FUDGE | "FUDGE: Controlled Text Generation With Future Discriminators" | https://arxiv.org/abs/2104.05218 |
| ACCESS | "Controllable Sentence Simplification" | https://arxiv.org/abs/1910.02677 |
| KeyBERT | "KeyBERT: Minimal Keyword Extraction" | https://github.com/MaartenGr/KeyBERT |
| YAKE | "YAKE! Keyword Extraction" | https://arxiv.org/abs/2001.11850 |
| LayoutLMv3 | "LayoutLMv3: Pre-training for Document AI" | https://arxiv.org/abs/2204.08387 |
| Donut | "OCR-free Document Understanding Transformer" | https://arxiv.org/abs/2111.15664 |
| Nougat | "Nougat: Neural Optical Understanding for Academic Documents" | https://arxiv.org/abs/2308.13418 |
| PAWS | "PAWS: Paraphrase Adversaries from Word Scrambling" | https://arxiv.org/abs/1904.01130 |

### Libraries
- **spaCy**: https://spacy.io/ industrial NLP, POS, NER, dep parsing
- **Stanza** (Stanford): https://stanfordnlp.github.io/stanza/ high-accuracy, 60+ languages
- **allennlp**: https://allennlp.org/ coreference, SRL, constituency parsing
- **fastcoref**: https://github.com/shon-otmazgin/fastcoref
- **REBEL**: https://github.com/Babelscape/rebel
- **KeyBERT**: https://github.com/MaartenGr/KeyBERT
- **YAKE**: https://github.com/LIAAD/yake
- **benepar** (constituency): https://github.com/nikitakit/self-attentive-parser

### Benchmarks
- **CoNLL-2003** (NER): https://aclanthology.org/W03-0419/
- **ACE 2005** (relation + event): LDC corpus
- **OntoNotes** (NER + coref + SRL): https://catalog.ldc.upenn.edu/LDC2013T19
- **PAWS** (paraphrase): https://github.com/google-research-datasets/paws
- **FUNSD / CORD** (document understanding): https://guillaumejaume.github.io/FUNSD/